# Глубинное обучение для текстовых данных, ФКН ВШЭ

## Домашнее задание 1: Text Suggestion

### Оценивание и штрафы

Максимально допустимая оценка за работу — 10 баллов. Сдавать задание после жесткого дедлайна нельзя. При сдачи решения после мягкого дедлайна за каждый день просрочки снимается по одному баллу.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом для пользоваться запрещается даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

__Мягкий дедлайн: 24 нояб

__Жесткий дедлайн: 27 нояб


### О задании

В этом задании вам предстоит реализовать систему, предлагающую удачное продолжение слова или нескольких следующих слов в режиме реального времени по типу тех, которые используются в телефонах, поисковой строке или приложении почты. Полученную систему вам нужно будет обернуть в пользовательский интерфейс с помощью библиотеки [reflex](https://github.com/reflex-dev/reflex), чтобы ей можно было удобно пользоваться, а так же, чтобы убедиться, что все работает как надо. В этот раз вам не придется обучать никаких моделей, мы ограничимся n-граммной генерацией.

### Структура

Это домашнее задание состоит из двух частей предположительно одинаковых по сложности. В первой вам нужно будет выполнить 5 заданий, по итогам которых вы получите минимально рабочее решение. А во второй, пользуясь тем, что вы уже сделали реализовать полноценную систему подсказки текста с пользовательским интерфейсом. Во второй части мы никак не будем ограничивать вашу фантазию. Делайте что угодно, лишь бы получилось в результате получился удобный фреймворк. Чем лучше у вас будет результат, тем больше баллов вы получите. Если будет совсем хорошо, то мы добавим бонусов сверху по своему усмотрению.

### Оценивание
При сдаче зададания в anytask вам будет необходимо сдать весь код, а также отчет с подробным описанием техник, которые в применили для создания вашей системы. Не лишним будет также написать и о том, что у вас не получилось и почему.

За часть с заданиями можно будет получить до __5__ баллов, за отчет – до __3__ баллов, 2 балл за доп вопросы, если возникнут, если вопросов не возникло, считаем, что 2 балла вы получили 

## Часть 1

### Данные

Для получения текстовых статистик используйте датасет `emails.csv`. Вы можете найти его по [ссылке](https://disk.yandex.ru/d/ikyUhWPlvfXxCg). Он содержит более 500 тысяч электронных писем на английском языке.

In [1]:
import pandas as pd

emails = pd.read_csv('emails.csv')
len(emails)

517401

In [2]:
emails

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...
...,...,...
517396,zufferli-j/sent_items/95.,Message-ID: <26807948.1075842029936.JavaMail.e...
517397,zufferli-j/sent_items/96.,Message-ID: <25835861.1075842029959.JavaMail.e...
517398,zufferli-j/sent_items/97.,Message-ID: <28979867.1075842029988.JavaMail.e...
517399,zufferli-j/sent_items/98.,Message-ID: <22052556.1075842030013.JavaMail.e...


Заметьте, что данные очень грязные. В каждом письме содержится различная мета-информация, которая будет только мешать при предсказании продолжения текста.

__Задание 1 (1 балл).__ Очистите корпус текстов по вашему усмотрению. В идеале обработанные тексты должны содержать только текст самого письма и ничего лишнего по типу ссылок, адресатов и прочих символов, которыми мы точно не хотим продолжать текст. Оценка будет выставляться по близости вашего результата к этому идеалу.

In [3]:
import re

In [4]:

cleaned_messages = []
new_emails_df = emails

#чистим текст через регулярку
for message in new_emails_df['message']:
    try:
        
        body = message.split("\n\n", 1)[-1]

        
        body = re.sub(
            r"(^Message-ID:.*?$)|(^From:.*?$)|(^To:.*?$)|(^Sent:.*?$)|(^Subject:.*?$)|"
            r"(^[-]{5,}.*?$)|(^Forwarded by.*?$)|"
            r"(^.*on \d{2}/\d{2}/\d{4}.*AM|PM$)|"
            r"(^.*@.*?$)|"  
            r"(^\d{2}:\d{2} (AM|PM) .*?$)|(^\tFrom:.*?$)|"
            r"(^Sent by:.*?$)|(^cc:.*?$)|(^Subject:.*?$)|"
            r"(\bhttps?://\S+)|(\bwww\.\S+)|(\b\S+\.\w{2,4}\b)|"  
            r"([\{\}\[\]/])",  
            "",
            body,
            flags=re.MULTILINE
        )

        
        body = re.sub(r"^>.*?$", "", body, flags=re.MULTILINE)

        
        body = re.sub(r"\n\s*\n", "\n", body).strip()

        
        cleaned_messages.append(body)
    except Exception as e:
        cleaned_messages.append(None)


new_emails_df['fully_cleaned_message'] = cleaned_messages

new_emails_df

,file,message,fully_cleaned_message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...,Here is our forecast
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...,Traveling to have a business meeting takes the...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...,test successful. way to go!!!
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...,"Randy,\n Can you send me a schedule of the sal..."
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...,Let's shoot for Tuesday at 11:45.
...,...,...,...
517396,zufferli-j/sent_items/95.,Message-ID: <26807948.1075842029936.JavaMail.e...,This is a trade with OIL-SPEC-HEDGE-NG (John L...
517397,zufferli-j/sent_items/96.,Message-ID: <25835861.1075842029959.JavaMail.e...,Some of my position is with the Alberta Term b...
517398,zufferli-j/sent_items/97.,Message-ID: <28979867.1075842029988.JavaMail.e...,"2\n -----Original Message-----\nMorning John,\..."
517399,zufferli-j/sent_items/98.,Message-ID: <22052556.1075842030013.JavaMail.e...,Analyst\t\t\t\t\tRank\nStephane Brodeur\t\t\t1...


In [5]:
new_emails_df[:1000].to_excel('mess3.xlsx')

Для следующего задания вам нужно будет токенизировать текст. Для этого просто разбейте его по словам. Очевидно, итоговый результат будет лучше, если ваша система также будет предлагать уместную пунктуацию. Но если вы считаете, что результат получается лучше без нее, то можете удалить все небуквенные символы на этапе токенизации.

In [6]:
tokenized_messages = []

#токенезируем текст, пунктуация игнорируется
for message in new_emails_df['fully_cleaned_message']:
    try:
        
        tokens = re.findall(r'\b\w+\b', message.lower())  
        tokenized_messages.append(tokens)
    except Exception as e:
        
        tokenized_messages.append([])


new_emails_df['tokens'] = tokenized_messages
new_emails_df

,file,message,fully_cleaned_message,tokens
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...,Here is our forecast,"[here, is, our, forecast]"
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...,Traveling to have a business meeting takes the...,"[traveling, to, have, a, business, meeting, ta..."
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...,test successful. way to go!!!,"[test, successful, way, to, go]"
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...,"Randy,\n Can you send me a schedule of the sal...","[randy, can, you, send, me, a, schedule, of, t..."
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...,Let's shoot for Tuesday at 11:45.,"[let, s, shoot, for, tuesday, at, 11, 45]"
...,...,...,...,...
517396,zufferli-j/sent_items/95.,Message-ID: <26807948.1075842029936.JavaMail.e...,This is a trade with OIL-SPEC-HEDGE-NG (John L...,"[this, is, a, trade, with, oil, spec, hedge, n..."
517397,zufferli-j/sent_items/96.,Message-ID: <25835861.1075842029959.JavaMail.e...,Some of my position is with the Alberta Term b...,"[some, of, my, position, is, with, the, albert..."
517398,zufferli-j/sent_items/97.,Message-ID: <28979867.1075842029988.JavaMail.e...,"2\n -----Original Message-----\nMorning John,\...","[2, original, message, morning, john, i, m, st..."
517399,zufferli-j/sent_items/98.,Message-ID: <22052556.1075842030013.JavaMail.e...,Analyst\t\t\t\t\tRank\nStephane Brodeur\t\t\t1...,"[analyst, rank, stephane, brodeur, 1, chad, cl..."


Очистим текст от мусора, который мы не очистили при первой итерации. Благодаря токенизации слов мы можем найти еще мусорные конструкции и удалить их из текста

In [7]:
filtered_tokens_list = []
updated_cleaned_messages = []

#удалим все словестные конструкции, в которых есть цифры и любые символы
for i in range(len(new_emails_df)):
    original_message = new_emails_df.loc[i, 'fully_cleaned_message']
    tokens = new_emails_df.loc[i, 'tokens']


    filtered_tokens = [token for token in tokens if token.isalpha()]
    filtered_tokens_list.append(filtered_tokens)


    if isinstance(original_message, str):
        updated_message = ' '.join(
            [word for word in re.findall(r'\b\w+\b', original_message) if word.isalpha()]
        )
    else:
        updated_message = None
    updated_cleaned_messages.append(updated_message)


new_emails_df['filtered_tokens'] = filtered_tokens_list
new_emails_df['updated_fully_cleaned_message'] = updated_cleaned_messages
new_emails_df

,file,message,fully_cleaned_message,tokens,filtered_tokens,updated_fully_cleaned_message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...,Here is our forecast,"[here, is, our, forecast]","[here, is, our, forecast]",Here is our forecast
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...,Traveling to have a business meeting takes the...,"[traveling, to, have, a, business, meeting, ta...","[traveling, to, have, a, business, meeting, ta...",Traveling to have a business meeting takes the...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...,test successful. way to go!!!,"[test, successful, way, to, go]","[test, successful, way, to, go]",test successful way to go
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...,"Randy,\n Can you send me a schedule of the sal...","[randy, can, you, send, me, a, schedule, of, t...","[randy, can, you, send, me, a, schedule, of, t...",Randy Can you send me a schedule of the salary...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...,Let's shoot for Tuesday at 11:45.,"[let, s, shoot, for, tuesday, at, 11, 45]","[let, s, shoot, for, tuesday, at]",Let s shoot for Tuesday at
...,...,...,...,...,...,...
517396,zufferli-j/sent_items/95.,Message-ID: <26807948.1075842029936.JavaMail.e...,This is a trade with OIL-SPEC-HEDGE-NG (John L...,"[this, is, a, trade, with, oil, spec, hedge, n...","[this, is, a, trade, with, oil, spec, hedge, n...",This is a trade with OIL SPEC HEDGE NG John La...
517397,zufferli-j/sent_items/96.,Message-ID: <25835861.1075842029959.JavaMail.e...,Some of my position is with the Alberta Term b...,"[some, of, my, position, is, with, the, albert...","[some, of, my, position, is, with, the, albert...",Some of my position is with the Alberta Term b...
517398,zufferli-j/sent_items/97.,Message-ID: <28979867.1075842029988.JavaMail.e...,"2\n -----Original Message-----\nMorning John,\...","[2, original, message, morning, john, i, m, st...","[original, message, morning, john, i, m, still...",Original Message Morning John I m still workin...
517399,zufferli-j/sent_items/98.,Message-ID: <22052556.1075842030013.JavaMail.e...,Analyst\t\t\t\t\tRank\nStephane Brodeur\t\t\t1...,"[analyst, rank, stephane, brodeur, 1, chad, cl...","[analyst, rank, stephane, brodeur, chad, clark...",Analyst Rank Stephane Brodeur Chad Clark Ian C...


In [25]:
new_emails_df[:10000].to_excel('mama.xlsx')

## Дополнение слова

Описанная система будет состоять из двух частей: дополнение слова до целого и генерация продолжения текста (или вариантов продолжений). Начнем с первой части.

В этой части вам предстоит реализовать метод дополнения слова до целого по его началу (префиксу). Для этого сперва необходимо научиться находить все слова, имеющие определенный префикс. Мы будем вызывать функцию поиска подходящих слов после каждой напечатанной пользователем буквы. Поэтому нам очень важно, чтобы поиск работал как можно быстрее. Простой перебор всех слов занимает $O(|V| \cdot n)$ времени, где $|V|$ – размер словаря, а $n$ – длина префикса. Мы же напишем [префиксное дерево](https://ru.wikipedia.org/wiki/Префиксное_дерево), которое позволяет искать слова за $O(n + m)$, где $m$ – число подходящих слов.

__Задание 2 (1 балл).__ Допишите префиксное дерево для поиска слов по префиксу. Ваше дерево должно работать за $O(n + m)$ операции, в противном случае вы не получите баллов за это задание.

In [8]:
from typing import List

class PrefixTreeNode:
    def __init__(self):
        # словарь с буквами, которые могут идти после данной вершины
        self.children: dict[str, PrefixTreeNode] = {}
        self.is_end_of_word = False

class PrefixTree:
    def __init__(self, vocabulary: List[str]):
        """
        vocabulary: список всех уникальных токенов в корпусе
        """
        self.root = PrefixTreeNode()
        
        for word in vocabulary:
            self.add_word(word)

    def add_word(self, word: str):
        """
        Добавление слова в префиксное дерево.
        """
        current_node = self.root

        for char in word:
            if char not in current_node.children:
                current_node.children[char] = PrefixTreeNode()
            current_node = current_node.children[char]

        current_node.is_end_of_word = True    

    def search_prefix(self, prefix) -> List[str]:
        """
        Возвращает все слова, начинающиеся на prefix
        prefix: str – префикс слова
        """

        current_node = self.root

        for char in prefix:
            if char in current_node.children:
                current_node = current_node.children[char]
            else:
                return []  #если отсутствует в дереве

        #возвращаем все слова с данного узла
        return self._collect_words(prefix, current_node)

    def _collect_words(self, prefix: str, node: PrefixTreeNode) -> List[str]:
        """
        Рекурсивно собирает все слова, начинающиеся с данного узла.
        """
        words = []
        if node.is_end_of_word:
            words.append(prefix)

        for char, child_node in node.children.items():
            words.extend(self._collect_words(prefix + char, child_node))

        return words

In [9]:
vocabulary = ['aa', 'aaa', 'abb', 'bba', 'bbb', 'bcd']
prefix_tree = PrefixTree(vocabulary)

assert set(prefix_tree.search_prefix('a')) == set(['aa', 'aaa', 'abb'])
assert set(prefix_tree.search_prefix('bb')) == set(['bba', 'bbb'])

Теперь, когда у нас есть способ быстро находить все слова с определенным префиксом, нам нужно их упорядочить по вероятности, чтобы выбирать лучшее. Будем оценивать вероятность слова по частоте его встречаемости в корпусе.

__Задание 3 (1 балл).__ Допишите класс `WordCompletor`, который формирует словарь и префиксное дерево, а так же умеет находить все возможные продолжения слова вместе с их вероятностями. В этом классе вы можете при необходимости дополнительно отфильтровать слова, например, удалив все самые редкие. Постарайтесь максимально оптимизировать ваш код.

In [10]:
import math
from collections import Counter
from typing import List, Tuple

class WordCompletor:
    def __init__(self, corpus):
        """
        corpus: list – корпус текстов
        """
        flattened_corpus = [word for sublist in corpus for word in sublist]

        #частотность слов
        self.word_counts = Counter(flattened_corpus)

        #общее кол-во слов
        self.total_words = sum(self.word_counts.values())

        self.prefix_tree = PrefixTree(list(self.word_counts.keys()))

    def get_words_and_probs(self, prefix: str) -> (List[str], List[float]):
        """
        Возвращает список слов, начинающихся на prefix,
        с их вероятностями (нормировать ничего не нужно)
        """
        #находим слова по префиксу
        words = self.prefix_tree.search_prefix(prefix)

        #считаем вероятность
        probs = [self.word_counts[word] / self.total_words for word in words]

        return words, probs

In [11]:
dummy_corpus = [
    ["aa", "ab"],
    ["aaa", "abab"],
    ["abb", "aa", "ab", "bba", "bbb", "bcd"],
]

word_completor = WordCompletor(dummy_corpus)
words, probs = word_completor.get_words_and_probs('a')
words_probs = list(zip(words, probs))
assert set(words_probs) == {('aa', 0.2), ('ab', 0.2), ('aaa', 0.1), ('abab', 0.1), ('abb', 0.1)}

In [12]:
words_probs

[('aa', 0.2), ('aaa', 0.1), ('ab', 0.2), ('abab', 0.1), ('abb', 0.1)]

## Предсказание следующих слов

Теперь, когда мы умеем дописывать слово за пользователем, мы можем пойти дальше и предожить ему несколько следующих слов с учетом дописанного. Для этого мы воспользуемся n-граммами и будем советовать n следующих слов. Но сперва нужно получить n-граммную модель.

Напомним, что вероятность последовательности для такой модели записывается по формуле
$$
P(w_1, \dots, w_T) = \prod_{i=1}^T P(w_i \mid w_{i-1}, \dots, w_{i-n}).
$$

Тогда, нам нужно оценить $P(w_i \mid w_{i-1}, \dots, w_{i-n})$ по частоте встречаемости n-граммы.   

__Задание 4 (1 балл).__ Напишите класс для n-граммной модели. Понятное дело, никакого сглаживания добавлять не надо, мы же не хотим, чтобы модель советовала случайные слова (хоть и очень редко).

In [13]:
from typing import List, Tuple
from collections import Counter, defaultdict
from itertools import islice

class NGramLanguageModel:
    def __init__(self, corpus, n): 
        """
        Инициализация n-граммной модели.
        corpus: список списков слов (корпус)
        n: длина n-грамм
        """
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        
        for sentence in corpus:
            for ngram_size in range(1, len(sentence) + 1):  #генерация нграмм разной длины
                for start_idx in range(len(sentence) - ngram_size + 1):  #перебор индексов
                    ngram = tuple(sentence[start_idx:start_idx + ngram_size])
                    self.ngram_counts[ngram] += 1
                    if len(ngram) > 1:  #учитываем только нграммы длиной > 1
                        context_key = ngram[:-1]
                        self.context_counts[context_key] += 1

    def get_next_words_and_probs(self, prefix: list) -> (List[str], List[float]):
        """
        Возвращает список слов, которые могут идти после prefix,
        а так же список вероятностей этих слов
        """

        prefix_tuple = tuple(prefix)
        context_count = self.context_counts.get(prefix_tuple, 0)
        if context_count == 0:
            return [], []

        next_word_candidates = {}
        for ngram, frequency in self.ngram_counts.items():
            if len(ngram) == len(prefix_tuple) + 1 and ngram[:-1] == prefix_tuple:
                following_word = ngram[-1]
                next_word_candidates[following_word] = frequency

        next_words = []
        probs = []
        for candidate, freq in next_word_candidates.items():
            probability = freq / context_count
            next_words.append(candidate)
            probs.append(probability)

        return next_words, probs


In [14]:
dummy_corpus = [
    ['aa', 'aa', 'aa', 'aa', 'ab'],
    ['aaa', 'abab'],
    ['abb', 'aa', 'ab', 'bba', 'bbb', 'bcd']
]

n_gram_model = NGramLanguageModel(corpus=dummy_corpus, n=2)

next_words, probs = n_gram_model.get_next_words_and_probs(['aa', 'aa'])
words_probs = list(zip(next_words, probs))

assert set(words_probs) == {('aa', 2/3), ('ab', 1/3)}

In [15]:
set(words_probs)

{('aa', 0.6666666666666666), ('ab', 0.3333333333333333)}

Отлично, мы теперь можем объединить два метода в автоматический дописыватель текстов: первый будет дополнять слово, а второй – предлагать продолжения. Хочется, чтобы предлагался список возможных продолжений, из который пользователь сможет выбрать наиболее подходящее. Самое сложное тут – аккуратно выбирать, что показывать, а что нет.   

__Задание 5 (1 балл).__ В качестве первого подхода к снаряду реализуйте метод, возвращающий всегда самое вероятное продолжение жадным способом. Если вы справитесь, то сможете можете добавить опцию поддержки нескольких вариантов продолжений, что сделает метод гораздо лучше.

In [16]:
from typing import Union

class TextSuggestion:
    def __init__(self, word_completor, n_gram_model):
        self.word_completor = word_completor
        self.n_gram_model = n_gram_model

    def suggest_text(self, text: Union[str, list], n_words=3, n_texts=1) -> list[list[str]]:
        """
        Возвращает возможные варианты продолжения текста (по умолчанию только один)
        
        text: строка или список слов – написанный пользователем текст
        n_words: число слов, которые дописывает n-граммная модель
        n_texts: число возвращаемых продолжений (пока что только одно)
        
        return: list[list[srt]] – список из n_texts списков слов, по 1 + n_words слов в каждом
        Первое слово – это то, которое WordCompletor дополнил до целого.
        """

        if isinstance(text, str):
            text = text.strip().split()  #токенезируем текст
        else:
            text = text[:]

        if not text:  #если нет текста, возвращаем пустой список
            return []

        suggestions = []

        for _ in range(n_texts):  #смотрим только один вариант продолжения
            current_text = text[:]
            last_word = current_text[-1]

            #выбираем последнее слово
            completions, probs = self.word_completor.get_words_and_probs(last_word)
            if completions:
                max_prob_index = probs.index(max(probs))
                current_text[-1] = completions[max_prob_index]

            #задаем контекст
            suggestion = [current_text[-1]]
            context = current_text[-(self.n_gram_model.n - 1):] if self.n_gram_model.n > 1 else []

            #генерим следующие слова
            for _ in range(n_words):
                next_words, next_probs = self.n_gram_model.get_next_words_and_probs(context)
                if not next_words:
                    break  #если нет предиктов то останавливаем цикл
                max_prob_index = next_probs.index(max(next_probs))
                next_word = next_words[max_prob_index]
                suggestion.append(next_word)

                #обновляем контекст
                if self.n_gram_model.n > 1:
                    context = context[1:] + [next_word] if len(context) >= self.n_gram_model.n - 1 else context + [next_word]
                else:
                    context = [next_word]

            suggestions.append(suggestion)

        return suggestions

In [17]:
dummy_corpus = [
    ['aa', 'aa', 'aa', 'aa', 'ab'],
    ['aaa', 'abab'],
    ['abb', 'aa', 'ab', 'bba', 'bbb', 'bcd']
]

word_completor = WordCompletor(dummy_corpus)
n_gram_model = NGramLanguageModel(corpus=dummy_corpus, n=3)
text_suggestion = TextSuggestion(word_completor, n_gram_model)

assert text_suggestion.suggest_text(['aa', 'aa'], n_words=3, n_texts=1) == [['aa', 'aa', 'aa', 'aa']]
assert text_suggestion.suggest_text(['abb', 'aa', 'ab'], n_words=2, n_texts=1) == [['ab', 'bba', 'bbb']]

In [18]:
text_suggestion.suggest_text(['aa', 'aa'], n_words=3, n_texts=1)

[['aa', 'aa', 'aa', 'aa']]

In [19]:
text_suggestion = TextSuggestion(word_completor, n_gram_model)

## Часть 2

Настало время довести вашу систему до ума. В этой части вы можете модифицировать все классы по своему усмотрению и добавлять любые эвристики. Если нужно, то дополнительно обрабатывать текст и вообще делать все, что считаете нужным, __кроме использования дополнительных данных__. Главное – вы должны обернуть вашу систему в пользовательский интерфейс с помощью [reflex](https://github.com/reflex-dev/reflex). В нем можно реализовать почти любой функционал по вашему желанию.

Мы настоятельно рекомендуем вам оформить код в проект, а не писать в ноутбуке. Но если вам очень хочется писать тут, то хотя бы не меняйте код в предыдущих заданиях, чтобы его можно было нормально оценивать.

При сдаче решения прикрепите весь ваш __код__, __отчет__ по второй части и __видео__ с демонстрацией работы вашей системы. Удачи!

# Описание работы

Решение задания доступно в данном репозитории: https://github.com/Spacelightpony/University/tree/Main/ADD%20Projects/NLP/HW1

В ходе выполнения проекта была интегрирована модель для генерации текста в UI reflex.

Так как приложение запускалось на локальной машине корпус данных после предварительно обработки пришлось сильно уменьшить для избежания проблемы с отсутствием оперативной памяти, что негативно сказалось на качестве генерированного текста.

Тем не менее, приложение работает так, как и было задумано.